# 🧩 Unit IV: Clustering & Association Rules

## 1. Executive Summary
This unit focuses on unsupervised learning to discover hidden customer segments and purchase patterns.

**Objectives:**
1.  **K-Means Clustering**: Segment customers into distinct groups based on RFM.
2.  **Hierarchical Clustering**: Visualize relationships using Dendrograms.
3.  **Association Rules (Apriori)**: Market Basket Analysis to find "Product A -> Product B" rules.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from sklearn.preprocessing import StandardScaler

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..')))
from src.models.clustering import ClusterAnalysis
from src.models.rules import MarketBasketAnalysis

# Load data
rfm = pd.read_csv('../data/processed/rfm_customer_data.csv')
df_trans = pd.read_csv('../data/processed/cleaned_transactions.csv') # For Apriori

## 2. Customer Segmentation (K-Means)
We use scaled RFM values.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

cluster_model = ClusterAnalysis()

# Elbow Method to find optimal K
scores = []
for k in range(2, 8):
    l, score, _ = cluster_model.kmeans_clustering(X_scaled, n_clusters=k)
    scores.append(score)

plt.figure(figsize=(8, 4))
plt.plot(range(2, 8), scores, marker='o')
plt.title('Silhouette Score vs K')
plt.show()

In [ ]:
# Applying K=3 based on typical segmentation needs
labels, score, centers = cluster_model.kmeans_clustering(X_scaled, n_clusters=3)
rfm['Cluster'] = labels
print(f"Silhouette Score: {score:.3f}")
print(rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean())

## 3. Hierarchical Clustering (Dendrogram)
Visualizing the hierarchy of clusters.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Sample for readability
sample_data = X_scaled[:50]
Z = linkage(sample_data, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(Z)
plt.title('Hierarchical Clustering Dendrogram (Sample=50)')
plt.xlabel('Customers')
plt.ylabel('Euclidean Distances')
plt.show()

## 4. Market Basket Analysis (Association Rules)
Identifying products frequently bought together using **Apriori Algorithm**.
*Note: We subset to top country for performance.*

In [ ]:
mba = MarketBasketAnalysis()
# Filter for UK and recent data for demo speed
subset_df = df_trans[(df_trans['Country'] == 'United Kingdom')].tail(1000)
basket = mba.prepare_basket(subset_df)

rules = mba.run_apriori(basket, min_support=0.02, min_confidence=0.1)
print(f"Rules Found: {len(rules)}")
if not rules.empty:
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False).head(5))

**📝 Insight:**
- **Lift > 1**: Indicates positive correlation. Products appear together more often than random chance.
- **Action**: Use these pairs for 'Frequently Bought Together' recommendations.